# Prepare model and data for the Responsible AI dashboard

The RAI dashboard needs:
1. Train and test data registered as **MLTable** data assets (with the label column)
2. A registered **MLflow model**

The model is trained in the cloud with the same `responsibleai-tabular` environment the RAI
components use, so the dashboard can load it without package-version conflicts.

In [1]:
"""
    Connect to the workspace (reads config.json in this folder)
"""
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print(ml_client.workspace_name)

Found the config file in: .\config.json


ws-azureml-practice


In [2]:
"""
    Load the heart dataset and split into train / test

    - Drop the 2 rows whose thal is a stray "1" / "2"
    - Keep thal as a string: the model one-hot encodes it, and the dashboard shows readable values
"""
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../cli/endpoints/batch/deploy-pipelines/training-with-components/data/train/heart.csv")
df = df[df["thal"].isin(["normal", "fixed", "reversible"])]

train_df, test_df = train_test_split(df, test_size=0.3, random_state=0, stratify=df["target"])
print(train_df.shape, test_df.shape)

(210, 14) (91, 14)


In [3]:
"""
    Write each split as parquet + an MLTable file that describes how to read it
"""
import os

mltable_yaml = """paths:
  - file: ./{name}.parquet
transformations:
  - read_parquet
"""

for name, split in [("train", train_df), ("test", test_df)]:
    folder = f"./data/heart-rai/{name}"
    os.makedirs(folder, exist_ok=True)
    split.to_parquet(f"{folder}/{name}.parquet", index=False)
    with open(f"{folder}/MLTable", "w") as f:
        f.write(mltable_yaml.format(name=name))

print(os.listdir("./data/heart-rai/train"))

['MLTable', 'train.parquet']


In [ ]:
"""
    Register both folders as MLTable data assets
"""
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

train_asset = ml_client.data.create_or_update(
    Data(name="heart-rai-train", path="./data/heart-rai/train", type=AssetTypes.MLTABLE,
         description="Heart disease training data for the RAI dashboard")
)
test_asset = ml_client.data.create_or_update(
    Data(name="heart-rai-test", path="./data/heart-rai/test", type=AssetTypes.MLTABLE,
         description="Heart disease test data for the RAI dashboard")
)
print(train_asset.id)
print(test_asset.id)

In [ ]:
"""
    Train the model in the cloud with the responsibleai-tabular environment
    (training script: rai_src/train.py)
"""
from azure.ai.ml import command, Input, Output

rai_env = "azureml://registries/azureml/environments/responsibleai-tabular/versions/28"

train_job = command(
    code="./rai_src",
    command=(
        "python train.py --train_data ${{inputs.train_data}} "
        "--target_column target --model_output ${{outputs.model_output}}"
    ),
    inputs={"train_data": Input(type=AssetTypes.MLTABLE, path=train_asset.id)},
    outputs={"model_output": Output(type=AssetTypes.MLFLOW_MODEL)},
    environment=rai_env,
    compute="aml-cluster",
    experiment_name="heart-rai-model-training",
    display_name="train-heart-model-for-rai",
)

returned_job = ml_client.jobs.create_or_update(train_job)
print("Monitor your job at", returned_job.studio_url)
# jobs.stream() fails on this workspace ("Incorrect padding" while reading logs),
# so poll the status instead. Logs are visible in the studio link above.
import time
while (status := ml_client.jobs.get(returned_job.name).status) not in ("Completed", "Failed", "Canceled"):
    print(status)
    time.sleep(30)
print(status)

In [ ]:
"""
    Register the job's output as an MLflow model
"""
from azure.ai.ml.entities import Model

model = ml_client.models.create_or_update(
    Model(
        name="heart-rai-model",
        path=f"azureml://jobs/{returned_job.name}/outputs/model_output",
        type=AssetTypes.MLFLOW_MODEL,
        description="GradientBoosting heart disease classifier for the RAI dashboard",
    )
)
print(model.id)